# Test Rebalancer

End-to-end test of rebalancing pipeline on top of `DataLoaderGraph` snapshots.

In [1]:
import pandas as pd
from IPython.display import display

from gbp.loaders import DataLoaderGraph, DataLoaderMock, GraphLoaderConfig
from gbp.rebalancer import DataLoaderRebalancer, Rebalancer, RebalancerConfig

## 1. Load graph data

In [2]:
mock = DataLoaderMock({"n": 8, "n_depots": 2, "n_timestamps": 48})
loader = DataLoaderGraph(mock, GraphLoaderConfig(distance_backend="haversine"))
loader.load_data()

print(f"Available dates: {len(loader.available_dates)}")
print(loader.available_dates[:3])

2026-03-08 14:58:59 [info     ] load_start                     loader=graph
2026-03-08 14:58:59 [debug    ] source_validated               loader=graph
2026-03-08 14:58:59 [info     ] load_done                      edges=90 loader=graph nodes=10
Available dates: 48
DatetimeIndex(['2025-01-01 00:00:00', '2025-01-01 01:00:00',
               '2025-01-01 02:00:00'],
              dtype='datetime64[ns]', freq='h')


## 2. Configure rebalancer

In [3]:
config = RebalancerConfig(
    inventory_node_type="station",
    depot_node_type="depot",
    min_threshold=0.3,
    max_threshold=0.7,
    time_limit_seconds=5,
)

dataloader_rebalancer = DataLoaderRebalancer(loader, config)
rebalancer = Rebalancer(dataloader_rebalancer, config)
config

RebalancerConfig(inventory_node_type='station', depot_node_type='depot', min_threshold=0.3, max_threshold=0.7, time_limit_seconds=5)

## 3. Find a snapshot with imbalance

In [4]:
selected_date = None

for d in loader.available_dates:
    dataloader_rebalancer.load_data(date=d)
    if dataloader_rebalancer.data is not None:
        selected_date = d
        break

if selected_date is None:
    print("No imbalanced snapshot was found in current mock data.")
else:
    print(f"Selected snapshot: {selected_date}")
    print(f"Sources: {(dataloader_rebalancer.df_node_demand['utilization'] > config.max_threshold).sum()}")
    print(f"Destinations: {(dataloader_rebalancer.df_node_demand['utilization'] < config.min_threshold).sum()}")

2026-03-08 14:59:06 [debug    ] snapshot                       date='2025-01-01 00:00:00' loader=graph
Selected snapshot: 2025-01-01 00:00:00
Sources: 4
Destinations: 1


## 4. Inspect node demand table

In [5]:
df_node_demand = dataloader_rebalancer.df_node_demand.copy()
display(df_node_demand.sort_values("utilization", ascending=False))

,node_id,latitude,longitude,inventory_capacity,quantity,utilization,target_count,balance
0,station_1,40.873688,-73.914443,44,43,0.977273,22.0,21.0
2,station_3,40.846835,-73.966424,20,19,0.950000,10.0,9.0
7,station_8,40.847872,-73.948521,45,42,0.933333,22.5,19.5
5,station_6,40.883226,-73.970493,24,20,0.833333,12.0,8.0
4,station_5,40.792148,-73.928487,36,20,0.555556,18.0,2.0
3,station_4,40.826059,-73.944477,39,18,0.461538,19.5,-1.5
1,station_2,40.750203,-73.953404,22,8,0.363636,11.0,-3.0
6,station_7,40.881722,-73.973431,33,2,0.060606,16.5,-14.5


## 5. Inspect generated PDP model

In [6]:
if dataloader_rebalancer.data is None:
    print("No PDP model: current snapshot has no source/destination pairs.")
else:
    pdp = dataloader_rebalancer.data
    print(f"num_resources={pdp['num_resources']}")
    print(f"num_pairs={len(pdp['pairs'])}")
    print(f"matrix_shape={pdp['distance_matrix'].shape}")
    print(f"node_layout_example={pdp['node_ids'][:6]}")
    display(pd.DataFrame(pdp['pairs']).head())

num_resources=3
num_pairs=1
matrix_shape=(3, 3)
node_layout_example=['depot', 'station_1_pickup', 'station_7_delivery']


,pickup_node_id,pickup_latitude,pickup_longitude,delivery_node_id,delivery_latitude,delivery_longitude,quantity
0,station_1,40.873688,-73.914443,station_7,40.881722,-73.973431,14


## 6. Run full rebalancing pipeline

In [7]:
if selected_date is None:
    print("Pipeline not executed because no imbalanced snapshot was found.")
else:
    rebalancer.run(date=selected_date)

2026-03-08 14:59:28 [debug    ] snapshot                       date='2025-01-01 00:00:00' loader=graph
Solution found — objective: 23916, total distance: 23916, dropped nodes: []


## 7. Route output

In [8]:
if hasattr(rebalancer, "route_df") and rebalancer.route_df is not None:
    display(rebalancer.route_df)
else:
    print("No route output (no feasible solution or no imbalances).")

,resource_id,step,node_id,action,quantity,cumulative_load
0,0,0,depot,depot,0,0
1,0,1,station_1,pickup,14,14
2,0,2,station_7,delivery,14,0
3,0,3,depot,depot,0,0


## 8. Inventory before/after

In [9]:
if hasattr(rebalancer, "df_updated") and rebalancer.df_updated is not None:
    cols = [
        "node_id",
        "inventory_capacity",
        "old_quantity",
        "quantity",
        "inventory_change",
        "utilization",
        "new_utilization",
    ]
    display(rebalancer.df_updated[cols].sort_values("inventory_change", ascending=False))
else:
    print("No updated inventory output.")

,node_id,inventory_capacity,old_quantity,quantity,inventory_change,utilization,new_utilization
6,station_7,33,2,16,14,0.060606,0.484848
1,station_2,22,8,8,0,0.363636,0.363636
3,station_4,39,18,18,0,0.461538,0.461538
2,station_3,20,19,19,0,0.950000,0.950000
5,station_6,24,20,20,0,0.833333,0.833333
4,station_5,36,20,20,0,0.555556,0.555556
7,station_8,45,42,42,0,0.933333,0.933333
0,station_1,44,43,29,-14,0.977273,0.659091


## 9. How many snapshots are rebalanceable?

In [10]:
stats = []

for d in loader.available_dates:
    dataloader_rebalancer.load_data(date=d)
    stats.append({
        "date": d,
        "has_pdp_model": dataloader_rebalancer.data is not None,
    })

df_stats = pd.DataFrame(stats)
print(df_stats["has_pdp_model"].value_counts())
display(df_stats.head(10))

2026-03-08 14:59:43 [debug    ] snapshot                       date='2025-01-01 00:00:00' loader=graph
2026-03-08 14:59:43 [debug    ] snapshot                       date='2025-01-01 01:00:00' loader=graph
2026-03-08 14:59:43 [debug    ] snapshot                       date='2025-01-01 02:00:00' loader=graph
2026-03-08 14:59:43 [debug    ] snapshot                       date='2025-01-01 03:00:00' loader=graph
2026-03-08 14:59:43 [debug    ] snapshot                       date='2025-01-01 04:00:00' loader=graph
2026-03-08 14:59:43 [debug    ] snapshot                       date='2025-01-01 05:00:00' loader=graph
2026-03-08 14:59:43 [debug    ] snapshot                       date='2025-01-01 06:00:00' loader=graph
2026-03-08 14:59:43 [debug    ] snapshot                       date='2025-01-01 07:00:00' loader=graph
2026-03-08 14:59:43 [debug    ] snapshot                       date='2025-01-01 08:00:00' loader=graph
2026-03-08 14:59:43 [debug    ] snapshot                       date='2025

,date,has_pdp_model
0,2025-01-01 00:00:00,True
1,2025-01-01 01:00:00,True
2,2025-01-01 02:00:00,True
3,2025-01-01 03:00:00,True
4,2025-01-01 04:00:00,True
5,2025-01-01 05:00:00,True
6,2025-01-01 06:00:00,True
7,2025-01-01 07:00:00,True
8,2025-01-01 08:00:00,True
9,2025-01-01 09:00:00,True
